# Frame detection — six-frame taxonomy, actors/events, robust evidence (v7)

Restores `frame_label` = NOT_FRAMED / SINGLE_FRAME / MULTIPLE_FRAMES; substantive frame(s) are stored in `frame_type`. Evidence-grounding failures no longer invalidate the frame classification.


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

# --- Same CC-IN2P3 / Huma-Num Ollama setup as the segmentation notebook ---
OLLAMA_MODEL = "qwen3.5:35b"
# OLLAMA_MODEL = "gemma3:27b"
# OLLAMA_MODEL = "llama3.3:70b"

_HOST_PORT_RE = re.compile(r"^[\w\.\-]+:\d+$")

def _resolve_ollama_host() -> str:
    val = os.environ.get("OLLAMA_HOST")
    source = "kernel environment"
    if not val:
        try:
            raw = subprocess.run(
                ["bash", "-lc", "module load ollama >/dev/null 2>&1; echo $OLLAMA_HOST"],
                capture_output=True, text=True, timeout=30,
            ).stdout
            lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
            candidate = lines[-1] if lines else ""
            if _HOST_PORT_RE.match(candidate):
                val = candidate
                source = "module load ollama (login shell)"
        except Exception:
            pass
    if not val:
        val = "127.0.0.1:11434"
        source = "default fallback"
    if not val.startswith(("http://", "https://")):
        val = f"http://{val}"
    print(f"OLLAMA_HOST resolved to {val} (source: {source})")
    return val

OLLAMA_HOST = _resolve_ollama_host()
OLLAMA_NUM_CTX = 32768
OLLAMA_THINK = False

# Input dataset for frame detection.
# Default: a sample drawn from segment_full and renamed sample_relevant.
# You may use either "sample_relevant" or "sample_relevant.csv".
INPUT_DATASET = "sample_debate"
LOCAL_DATA_DIR = Path("./data")
LOCAL_OUTPUT_DIR = Path("./data")

def resolve_input_csv(name, data_dir=LOCAL_DATA_DIR):
    p = Path(str(name))
    if p.suffix.lower() != ".csv":
        p = p.with_suffix(".csv")
    # If only a filename was supplied, prefer ./data/<name>.csv, then ./<name>.csv.
    if p.parent == Path("."):
        candidates = [Path(data_dir) / p.name, p]
    else:
        candidates = [p]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    # Return the preferred location so the assertion below gives a useful path.
    return candidates[0]

LOCAL_INPUT_CSV = resolve_input_csv(INPUT_DATASET)
DATASET_STEM = LOCAL_INPUT_CSV.stem

# Number of NEW segment rows processed in this invocation. None = all remaining.
BATCH_SIZE = 100
CHECKPOINT_EVERY = 10
MAX_TOKENS = 3200
TEMPERATURE = 0.0
MAX_CHARS = 20000
MAX_RETRIES = 2
MAX_EVIDENCE_CHARS = 180

# False = foreground; True = detached process with logfile, useful for long runs.
RUN_IN_BACKGROUND = True
WORKER_SCRIPT_PATH = Path("./frame_detection_articles.py")

def model_slug(model: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", model)

def output_paths(model=OLLAMA_MODEL, dataset_stem=DATASET_STEM, out_dir=LOCAL_OUTPUT_DIR):
    stem = f"frames_{dataset_stem}_{model_slug(model)}"
    out_dir = Path(out_dir)
    return {
        "jsonl": out_dir / f"{stem}.jsonl",
        "json": out_dir / f"{stem}.json",
        "csv": out_dir / f"{stem}.csv",
        "log": out_dir / f"{stem}.log",
    }

print(f"Input CSV: {LOCAL_INPUT_CSV}")
print(output_paths())

In [ ]:
FRAME_DETECTION_PY = r"""
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import hashlib
import json
import os
import re
import time
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import requests

FRAME_NAMES = ["modernization", "patriotism", "victimhood", "moral", "economic", "diplomacy"]
POLARITIES = ["positive", "negative"]
FRAME_TYPES = {f"{frame} ({polarity})" for frame in FRAME_NAMES for polarity in POLARITIES}
FRAME_LABELS = {"NOT_FRAMED", "SINGLE_FRAME", "MULTIPLE_FRAMES"}
NOT_FRAMED = "NOT_FRAMED"

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")
if not OLLAMA_HOST.startswith(("http://", "https://")):
    OLLAMA_HOST = "http://" + OLLAMA_HOST
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen3.5:35b")
OLLAMA_NUM_CTX = int(os.environ.get("OLLAMA_NUM_CTX", "32768"))
OLLAMA_THINK = os.environ.get("OLLAMA_THINK", "false").lower() in ("1", "true", "yes")
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", "3200"))
TEMPERATURE = float(os.environ.get("TEMPERATURE", "0.0"))
MAX_CHARS = int(os.environ.get("MAX_CHARS", "20000"))
CHECKPOINT_EVERY = int(os.environ.get("CHECKPOINT_EVERY", "10"))
MAX_RETRIES = int(os.environ.get("MAX_RETRIES", "2"))
MAX_EVIDENCE_CHARS = int(os.environ.get("MAX_EVIDENCE_CHARS", "180"))

SYSTEM_PROMPT = r'''You are a social-science coding assistant for historical Chinese media analysis.

Your task is to apply a predefined media-frame codebook to the supplied TITLE + SEGMENT_TEXT and determine which, if any, narrative frames are substantively present.

Use a social-science concept of framing inspired by Goffman (1974) and Entman (1993). Use only TITLE + SEGMENT_TEXT. Do not supply interpretations from outside historical knowledge, stereotypes, or general associations with study abroad.

TWO MANDATORY GATES FOR EACH CANDIDATE FRAME

Apply both gates separately to EVERY proposed frame family and polarity. Passing them for one frame does not establish any other frame. The presence of students somewhere in the text is not a passage-wide relevance license.

GATE 1: SUBJECT RELEVANCE

Identify the specific object of the candidate interpretation: who or what is evaluated, explained, blamed, praised, or constructed as a victim, threat, resource, agent, or representative?

That interpretation must concern at least one eligible research subject:
- overseas students, returned students, or foreign-educated professionals, individually or collectively;
- their behavior, motivations, experiences, training, achievements, failures, or actions;
- study abroad as a practice or educational, social, political, economic, or cultural enterprise;
- policies, programs, institutions, or administrative practices directly governing study abroad or specifically affecting those subjects in their capacity as overseas/returned students or foreign-educated professionals.

The link must be established in the supplied text. Clear local reference (such as a pronoun referring to previously identified students) can establish it; repeating the word students in every sentence is unnecessary. Do not infer foreign education from a person's occupation, name, reputation, or historical identity.

A policy/institution qualifies only when the PARTICULAR interpretation concerns its study-abroad function or its direct effects on the eligible subjects. A ministry, university, government, or foreign power mentioned in a study-abroad article is not automatically eligible for all evaluations of it. General national education, corruption, poverty, oppression, reform, activism, or international relations remain context unless their specific interpretation is explicitly linked to the research subject.

Keep the source/speaker of an interpretation distinct from its target. Students can criticize officials, discuss national suffering, or report foreign affairs without themselves being framed as corrupt, victims, patriotic, or diplomatic. Do not transfer a speaker's interpretation of someone else onto the speaker, or a contextual frame onto students.

Family-specific target restrictions still apply: moral evaluates the conduct/character of eligible people; patriotism evaluates their loyalty/service or the national-duty meaning of study abroad; victimhood requires eligible people themselves to be the sympathetic victims or threatening predators. A directly related policy may supply the cause/context for these interpretations, but it cannot replace the required human target.

If the target or the specific link cannot be established, reject that candidate frame. Continue scanning the other families.

GATE 2: INTERPRETIVE FRAMING

After subject relevance is established, ask whether the text selectively emphasizes the eligible subject in a way that steers readers toward a particular interpretation, evaluation, causal understanding, or normative judgment in this frame family.

Relevance alone is insufficient. Framing is NOT:
- a keyword match or mere factual mention of a relevant topic;
- emotional wording without the family-specific interpretation;
- a neutral administrative, statistical, biographical, or descriptive statement;
- an interpretation supplied by the coder rather than the text.

Require an identifiable evaluative or interpretive claim, not necessarily an explicit adjective. A short passage or factual account can qualify when its wording, causal attribution, or selection clearly constructs investment/burden, reform success/failure, virtue/vice, national duty, sympathetic suffering/threat, or international mediation/influence. Length, repetition, and dominance are not prerequisites. Neutral facts must not be promoted into frames simply because they could be interpreted that way.

Evidence must support the specific interpretation and its relation to the eligible subject. If the relation and interpretation occur in separate passages, use separate exact evidence objects where needed; never fabricate a combined quote. Reject an unsupported candidate rather than retaining it with low confidence.

INDEPENDENT SIX-FAMILY SCAN

Evaluate modernization, patriotism, victimhood, moral, economic, and diplomacy independently. For EACH family:
1. Identify its specific target and establish the textual link required by Gate 1.
2. Identify the family-specific interpretive claim required by Gate 2.
3. Determine its polarity from that claim, and locate exact supporting evidence.

Do not stop after the most obvious frame. Do not rank frames as dominant or secondary. Include a secondary frame whenever it independently passes both gates; do not require it to organize the entire passage.

Multiple labels require independent textual support for EACH interpretation, not different names for the same unsupported inference. Evidence spans may overlap if they genuinely support distinct claims; separate quotations are not mechanically required. A modernization contribution is not automatically economic value or patriotism, financial hardship is not automatically victimhood, and misconduct is not automatically failed modernization or disloyalty.

Determine family presence before polarity. Assess polarity separately for each supported interpretation; do not give every frame the passage's overall tone. Both polarities within a family are permitted only if distinct positive and negative interpretations are independently supported. Deduplicate identical frame_type values.

FRAME TAXONOMY

1. modernization

CORE QUESTION:
Is study abroad interpreted as an enterprise of national modernization, reform, or progress, or are eligible people interpreted as its agents, pioneers, contributors, or failed agents?

POSITIVE:
Study abroad or foreign-educated people are represented as advancing national reform, reconstruction, institutional improvement, scientific/technological progress, educational transformation, or national capacity-building. Foreign knowledge or expertise qualifies only when its contribution to such progress is substantively established. An advocated or anticipated modernization role can qualify without a completed outcome, when positively framed.

NEGATIVE:
The text interprets study abroad or eligible people as failing that modernization task: disappointed reform expectations, ineffective contribution, inability to apply foreign training to national needs, or a mismatch that frustrates national progress. Criticism of the system can qualify when it specifically concerns study abroad's failure as a modernization enterprise.

EXCLUDE:
Education, credentials, technical expertise, appointments, personal career success, or travel abroad alone; general national backwardness or educational reform; financial costs or student misconduct without an independent national-reform/progress interpretation. Do not turn all praise/criticism of students into modernization.

POLARITY:
A demand for reform does not itself establish positive modernization. A diagnosis that the current study-abroad system fails national reform supports negative modernization; add positive only if an independently supported positive modernization role or promise is also expressed.

2. patriotism

CORE QUESTION:
Are eligible people's loyalty, national duty, sacrifice, or service to their country evaluated, or is study abroad explicitly interpreted as national service?

POSITIVE:
Students/returnees/foreign-educated professionals are portrayed as loyal, sacrificing for the nation, serving national salvation, defending national interests, or undertaking foreign study with a patriotic purpose.

NEGATIVE:
They are portrayed as disloyal, abandoning national duty, betraying their country, or failing an explicitly patriotic obligation.

EXCLUDE:
Any national activism, protest, war, government appointment, reference to China, or patriotic rhetoric by others without an interpretation of the eligible subjects' loyalty/service. Students' attendance at a political meeting is insufficient by itself. Their discussion of national oppression does not automatically praise their patriotism. Careerism or selfishness is not disloyalty unless the text establishes the national-duty judgment.

BOUNDARY:
Loyalty presented as personal ethical character can support moral; loyalty/service to the nation supports patriotism. Include both only when both interpretations are established, not because loyalty is a shared keyword. Modernization outcomes do not automatically establish patriotic motivation.

3. victimhood

CORE QUESTION:
Are eligible people THEMSELVES constructed as sympathetic victims/vulnerable sufferers, or as threatening predators/exploiters?

POSITIVE:
They are portrayed sympathetically as victims or vulnerable sufferers of discrimination, repression, unfair treatment, financial hardship, war, disaster, foreign hostility, or another external condition. The presentation must invite sympathy or establish vulnerability/injustice, not merely record an adverse event.

NEGATIVE:
They are represented as predators, exploiters, privileged threats, or harmful competitors victimizing others, in a way that invites outrage against them. Mere misconduct or ordinary competition does not establish this threatening/predatory construction.

EXCLUDE:
The nation, Chinese people generally, unrelated workers, officials, or other groups as victims; students merely speaking about, witnessing, or responding to their suffering; harmful policies without a supported construction of students as victims; any bad event automatically treated as victimhood.

POLARITY:
Positive means sympathetic student victims, even when the event is tragic. Negative means the students/returnees themselves are the threat/predator, not that their suffering is severe. Never transfer an oppressor's role onto the students it harms.

4. moral

CORE QUESTION:
Are the conduct or character of students, returnees, or foreign-educated professionals morally evaluated?

POSITIVE:
Praise of their diligence, discipline, honesty, integrity, selflessness, dedication, responsible conduct, public spirit, or other virtues. Loyalty can qualify when evaluated as personal moral character.

NEGATIVE:
Criticism of their indiscipline, misconduct, disobedience, dishonesty, corruption, selfishness, opportunism, careerism, status-seeking, or neglect of learning for personal advancement. Cultural alienation qualifies only when explicitly construed as their moral degradation.

EXCLUDE:
Unrelated corrupt officials, general social moral decline, students denouncing others' corruption, and institutional misconduct alone. Misuse of study-abroad funds by administrators can support economic but does not itself evaluate student/returnee character. An official can be an eligible moral target only if the text establishes that official as a returnee/foreign-educated professional and directs the moral evaluation at that person.

BOUNDARY:
Poor outcomes or unsuitable training are not necessarily moral failings. Do not infer moral blame from failure to modernize, financial expense, poverty, or foreign influence. Do not infer negative patriotism from selfish ambition without an explicit national-duty interpretation.

5. economic

CORE QUESTION:
Is study abroad or an eligible subject interpreted through investment, economic value, costs, financial burdens, resources, funding allocation, affordability, or financial hardship?

POSITIVE:
Study abroad is a productive investment or beneficial expenditure; foreign-trained people are economically valuable human capital; resources are allocated beneficially/efficiently; their training produces identifiable material/economic benefits.

NEGATIVE:
Study abroad imposes excessive costs or burdens on the state, society, families, or students; expenditure is wasted, diverted, inefficient, unfairly allocated, or yields poor financial returns; insufficient resources or unaffordable tuition/living/travel costs obstruct students' studies or survival.

EXCLUDE:
Neutral scholarship/stipend amounts, tuition schedules, budgets, salaries, or appointments; general national economic conditions; unrelated government corruption; modernization via science/industry without a substantive economic-value or resource interpretation. Personal ambition is not automatically economic framing.

BOUNDARY:
Modernization concerns national reform/progress through study abroad. Economic concerns investment, financial/material value, costs, and resources. Neither implies the other. Financial hardship supports economic (negative) when scarcity/affordability/burden is substantively emphasized; additionally code victimhood (positive) only when students are independently constructed as sympathetic sufferers.

6. diplomacy

CORE QUESTION:
Is study abroad or an eligible subject substantively interpreted as performing cross-border representation, cultural mediation, international advocacy, or a channel/target of foreign influence?

POSITIVE:
Eligible people/study abroad are portrayed as cultural ambassadors, international representatives or intermediaries, brokers of mutual understanding, or advocates representing national interests across international audiences/institutions. The representative or mediating function must be established.

NEGATIVE:
Study abroad or eligible people are portrayed as instruments or targets of foreign imperial influence, cultural penetration, domination, manipulation, or alienating foreign influence. The text must connect that influence to study abroad or the eligible subjects, not merely describe imperialism generally.

EXCLUDE:
Foreign travel, enrollment abroad, international contact, foreign recipients, language study, or exposure to foreign ideas alone; unrelated state diplomacy or international conflict; general cultural anxiety without the specific link. Writing to a foreign institution is insufficient unless its representative, advocacy, mediating, or influence function is substantively emphasized.

BOUNDARY:
Patriotism concerns national loyalty/service; diplomacy concerns the actual cross-border representative/mediating/influence function. A student action can support both with independent evidence. Foreign influence does not automatically establish students' immorality, disloyalty, or victimhood.

CONTRASTIVE CODING EXAMPLES

These are illustrative codebook examples, NOT input evidence. Never quote them in the output unless the identical text actually occurs in TITLE or SEGMENT_TEXT. For each example, assign only the interpretations shown by that example's wording.

- 留學生赴法入學。 → NOT_FRAMED: eligible subject, but only a neutral enrollment fact.
- 中國人民受帝國主義壓迫，留學生集會討論。 → No student victimhood; no automatic patriotism or diplomacy. The victims are Chinese people generally, and the students' meeting has no independently stated framing interpretation.
- 留美學生發表宣言，譴責國內官員貪污。 → No moral frame about students; the corrupt targets are unrelated officials. Student authorship does not transfer the accusation to students or automatically establish patriotism.
- 留學經費被官員侵吞，致學生無力繳納學費。 → economic (negative): study-abroad resources are diverted and affordability is obstructed; no student moral frame. Add victimhood only if the text also sympathetically constructs students as victims.
- 歸國學生以所學推動國家教育改革，為革新之先驅。 → modernization (positive); no automatic economic, moral, or patriotism label.
- 留學生所學不合國情，不能推動國家改革。 → modernization (negative); no automatic moral blame or financial burden.
- 留學生每月領津貼五十元。 → NOT_FRAMED: neutral funding amount.
- 留學耗費巨款，實為國家財政之重負。 → economic (negative); no automatic modernization failure.
- 留學投入可收豐厚經濟之利，為有益之投資。 → economic (positive); no automatic national reform frame.
- 留學生不務正業，終日追求官職。 → moral (negative); no automatic economic, modernization, or patriotism frame.
- 留學生不務正業，徒耗國家經費，歸國後亦不能推動改革。 → moral (negative), economic (negative), modernization (negative): distinct misconduct, resource waste, and failed reform claims.
- 留學生参加政治集會。 → NOT_FRAMED: political participation alone.
- 留學生犧牲個人利益，以救國為己任。 → patriotism (positive); add moral only if an independently established character/ethical interpretation supports it.
- 留學生背棄祖國，甘為敵國效力。 → patriotism (negative); no automatic diplomacy absent a substantive foreign-influence/representation interpretation.
- 留學生遭無理拘禁，孤苦無援，令人同情。 → victimhood (positive): sympathetic student victims, despite the negative event.
- 歸國學生憑特權排擠同胞，成為侵奪民眾生計之禍害。 → victimhood (negative): returnees constructed as predatory threats; do not add economic merely because livelihoods are mentioned unless their economic/resource dimension is independently framed.
- 留學生因津貼斷絕無力續學，流落異鄉，實堪憐憫。 → economic (negative) AND victimhood (positive): financial obstruction and explicit sympathetic suffering.
- 留學生向外國大學寄信詢問入學日期。 → NOT_FRAMED: foreign recipient alone is not diplomacy.
- 留學生向外國公眾闡明中國立場，充任兩國文化交流之橋樑。 → diplomacy (positive); add patriotism only with independent national-service/loyalty emphasis.
- 外國勢力藉留學教育操縱學生，使之成為文化侵略之工具。 → diplomacy (negative): foreign influence specifically through study abroad; no automatic student moral, patriotism, or victimhood label.

FINAL DECISION CHECK

Before producing the answer, perform these checks internally:
1. Reconsider ALL SIX families independently, including less salient economic and diplomacy interpretations. Do not stop at the strongest frame.
2. For EVERY proposed frame_type, identify its eligible target, textual link, family-specific interpretation, polarity, and exact evidence. Verify both gates independently; remove candidates supported only by context, keywords, outside knowledge, or another frame.
3. Check that national victims, unrelated corrupt officials, speakers, and contextual institutions have not supplied frames transferred to students.
4. Keep modernization (national reform/progress), economic (investment/cost/resources), and moral (eligible people's conduct/character) distinct. Include multiple labels only for independently supported claims.
5. Check student loyalty/service for patriotism, students as sympathetic victims or threatening predators for victimhood, and an actual representative/mediating/foreign-influence function linked to study abroad for diplomacy.
6. Check whether financial hardship independently supports economic (negative) and victimhood (positive); neither requires nor excludes the other.
7. Determine each polarity from the specific interpretation, not the overall tone or pleasantness of the event. Every selected polarity needs support.
8. Include every independently qualifying frame without dominance ranking; remove duplicate frame_type values. Never force a frame when neither gate is satisfied.

FRAME LABEL

Derive frame_label only from the final frame_type list:

- no frame_type → NOT_FRAMED
- exactly one frame_type → SINGLE_FRAME
- two or more frame_type values → MULTIPLE_FRAMES

frame_type may contain only:
- modernization (positive)
- modernization (negative)
- patriotism (positive)
- patriotism (negative)
- victimhood (positive)
- victimhood (negative)
- moral (positive)
- moral (negative)
- economic (positive)
- economic (negative)
- diplomacy (positive)
- diplomacy (negative)

FRAME EXPLANATION

Always provide a concise analytical justification for the classification.

For SINGLE_FRAME:
- explain why the selected frame meets the framing threshold;
- explain why its polarity applies.

For MULTIPLE_FRAMES:
- justify EACH selected frame_type separately;
- explain the textual emphasis supporting each frame and why its polarity applies;
- do not rank the frames as dominant or secondary.

For NOT_FRAMED:
- explain why potentially suggestive content remains descriptive, factual, administrative, biographical, or otherwise insufficient to meet the framing threshold.

The explanation must distinguish substantive framing from mere topical relevance. For each selected frame, identify the eligible subject and the specific interpretation that passes both gates; do not treat a speaker as the target by default. For NOT_FRAMED, explain whether subject relevance, interpretive framing, or both are absent.

ACTOR AND EVENT EXTRACTION

framing_actors:
The actor(s) explicitly presented as voicing, advancing, reporting, endorsing, or otherwise being responsible for the detected framing interpretation.

framed_actors:
The actor(s) or group(s) that are the subject or target of the detected framing interpretation.

framed_events:
The event(s), policy/policies, process(es), action(s), or situation(s) substantively interpreted through the detected frame(s).

For all three fields:
- extract only information supported by TITLE or SEGMENT_TEXT;
- every value other than "unspecified" must be copied as a short contiguous span from the original text;
- preserve original Chinese, historical characters, and OCR spelling exactly;
- do not translate, romanize, paraphrase, modernize characters, summarize, or infer;
- do not list every entity or event mentioned: include only those tied to the detected framing;
- return ["unspecified"] when no suitable source span can be identified;
- for NOT_FRAMED, return [].

FRAME EVIDENCE

For EVERY detected frame_type, provide at least one short piece of evidence supporting that specific frame.

Each evidence quote must:
- be copied VERBATIM from TITLE or SEGMENT_TEXT;
- be ONE CONTIGUOUS source span;
- preserve original Chinese, historical characters, and OCR spelling exactly;
- demonstrate the framing emphasis rather than merely contain a topic keyword.

Never insert "...", "…", brackets, translations, explanatory material, or text not present in the source.

If two separate passages are needed, return two separate evidence objects rather than combining them.

For NOT_FRAMED:
frame_evidence = []

CONFIDENCE

framing_confidence must be a number from 0.0 to 1.0 representing confidence in the overall classification.

Use lower confidence when:
- the framing distinction is genuinely borderline;
- polarity is ambiguous;
- the text is severely OCR-degraded;
- or evidence for one or more selected frames is weak.

Do not lower confidence merely because multiple frames are present.

PROCESSING ERROR

frame_processing_error is a pipeline-status field, NOT a substantive judgment.

Return false for every normally completed classification, including NOT_FRAMED and ambiguous cases.

OUTPUT

Return ONLY one valid JSON object with exactly these keys:

{
  "frame_label": "NOT_FRAMED | SINGLE_FRAME | MULTIPLE_FRAMES",
  "frame_type": [
    "modernization (positive)"
  ],
  "frame_explanation": "Concise English justification addressing every selected frame_type and its polarity.",
  "frame_evidence": [
    {
      "frame_type": "modernization (positive)",
      "quote": "verbatim contiguous source excerpt"
    }
  ],
  "framing_actors": [
    "verbatim source span or unspecified"
  ],
  "framed_actors": [
    "verbatim source span or unspecified"
  ],
  "framed_events": [
    "verbatim source span or unspecified"
  ],
  "framing_confidence": 0.0,
  "frame_processing_error": false
}'''


def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


def clean_text(x: Any) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return str(x).strip()


def row_id(row: pd.Series) -> str:
    doc_id = clean_text(row.get("doc_id", row.get("DocId", "")))
    seg_idx = clean_text(row.get("segment_index", ""))
    if doc_id and seg_idx:
        return f"{doc_id}::seg{seg_idx}"
    payload = "\n".join([doc_id, seg_idx, clean_text(row.get("title", "")), clean_text(row.get("segment_text", ""))])
    return "sha1:" + hashlib.sha1(payload.encode("utf-8")).hexdigest()


def extract_json(text: str) -> Dict[str, Any]:
    text = text.strip()
    if not text:
        raise ValueError("Empty model response")
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        raise ValueError("No JSON object found in model response")
    obj = json.loads(m.group(0))
    if not isinstance(obj, dict):
        raise ValueError("Parsed JSON is not an object")
    return obj


def normalize_label(x: Any) -> str:
    s = clean_text(x).lower()
    s = re.sub(r"\s+", " ", s)
    aliases = {
        "nationalism (positive)": "patriotism (positive)",
        "nationalism (negative)": "patriotism (negative)",
        "nationalism_patriotism (positive)": "patriotism (positive)",
        "nationalism_patriotism (negative)": "patriotism (negative)",
        "not framed": NOT_FRAMED,
        "not_framed": NOT_FRAMED,
    }
    if s in aliases:
        return aliases[s]
    if s == NOT_FRAMED.lower():
        return NOT_FRAMED
    return s


def _ground_extractive_span(value: Any, source_text: str) -> str | None:
    # Return an exact source substring for an actor/event candidate, or None.
    # Matching may ignore punctuation/spacing and a small set of historical
    # variants, but SAVED output is always sliced from the original source.
    text = clean_text(value)
    if not text or text.lower() == "unspecified":
        return None
    if text in source_text:
        return text
    stripped = text.strip('"\'“”‘’「」『』')
    if stripped in source_text:
        return stripped
    nq, _ = _match_normalize_with_map(stripped)
    ns, mapping = _match_normalize_with_map(source_text)
    if nq:
        j = ns.find(nq)
        if j >= 0:
            return source_text[mapping[j]:mapping[j + len(nq) - 1] + 1]
    return None


def normalize_actor_list(value: Any, field_name: str, source_text: str, require_unspecified: bool = False) -> List[str]:
    if require_unspecified:
        return ["unspecified"]
    if isinstance(value, str): value = [value]
    if not isinstance(value, list) or not value: return ["unspecified"]
    actors: List[str] = []
    for raw in value:
        actor = clean_text(raw)
        if not actor or actor.lower() == "unspecified": continue
        grounded = _ground_extractive_span(actor, source_text)
        if grounded and grounded not in actors: actors.append(grounded)
    return actors or ["unspecified"]


def normalize_event_list(value: Any, source_text: str, require_unspecified: bool = False) -> List[str]:
    if require_unspecified:
        return ["unspecified"]
    if isinstance(value, str): value = [value]
    if not isinstance(value, list) or not value: return ["unspecified"]
    events: List[str] = []
    for raw in value:
        event = clean_text(raw)
        if not event or event.lower() == "unspecified": continue
        grounded = _ground_extractive_span(event, source_text)
        if grounded and grounded not in events: events.append(grounded)
    return events or ["unspecified"]


def extraction_needs_repair(obj: Dict[str, Any], source_text: str) -> bool:
    for field in ("framing_actors", "framed_actors", "framed_events"):
        value = obj.get(field, ["unspecified"])
        if isinstance(value, str): value = [value]
        if not isinstance(value, list): return True
        for raw in value:
            text = clean_text(raw)
            if not text or text.lower() == "unspecified": continue
            if _ground_extractive_span(text, source_text) is None: return True
    return False


def _canonical_match_char(ch: str) -> str:
    variants={"敎":"教","僞":"偽","爲":"為","裏":"裡","臺":"台","衞":"衛","羣":"群"}
    return variants.get(ch,ch)

def _match_normalize_with_map(text: str) -> tuple[str, List[int]]:
    import unicodedata
    chars=[]; idx=[]
    for i,ch in enumerate(text):
        cat=unicodedata.category(ch)
        if ch.isspace() or cat.startswith("P") or cat.startswith("Z"): continue
        chars.append(_canonical_match_char(ch)); idx.append(i)
    return "".join(chars),idx

def repair_evidence_quote(quote: str, source_text: str) -> List[str]:
    quote=clean_text(quote)
    if not quote: return []
    if quote in source_text: return [quote]
    stripped=quote.strip('"\'“”‘’「」『』')
    if stripped in source_text: return [stripped]
    nq,_=_match_normalize_with_map(stripped); ns,mapping=_match_normalize_with_map(source_text)
    if nq:
        j=ns.find(nq)
        if j>=0: return [source_text[mapping[j]:mapping[j+len(nq)-1]+1]]
    parts=[p.strip() for p in re.split(r'(?:\.\.\.|…+|⋯+)',stripped) if p.strip()]
    if len(parts)>=2:
        found=[]
        for part in parts:
            np,_=_match_normalize_with_map(part); j=ns.find(np) if np else -1
            if j>=0: found.append(source_text[mapping[j]:mapping[j+len(np)-1]+1])
        if len(found)==len(parts): return found
    return []

def validate_and_repair(obj: Dict[str, Any], source_text: str) -> Dict[str, Any]:
    required={"frame_type","frame_explanation","framing_actors","framed_actors","framed_events","framing_confidence"}
    missing=sorted(required-set(obj.keys()))
    if missing: raise ValueError(f"Missing required model fields: {missing}")
    raw=obj.get("frame_type")
    if isinstance(raw,str): raw=[raw]
    if not isinstance(raw,list): raise ValueError("frame_type must be a list")
    types=[]
    for x in raw:
        t=normalize_label(x)
        if t==NOT_FRAMED: continue
        if t not in FRAME_TYPES: raise ValueError(f"Invalid frame type: {x!r}")
        if t not in types: types.append(t)
    order=[f"{f} ({p})" for f in FRAME_NAMES for p in POLARITIES]
    types=[t for t in order if t in types]
    label=NOT_FRAMED if not types else ("SINGLE_FRAME" if len(types)==1 else "MULTIPLE_FRAMES")
    explanation=clean_text(obj.get("frame_explanation"))
    if not explanation: raise ValueError("frame_explanation is empty")
    try: conf=float(obj.get("framing_confidence"))
    except Exception as e: raise ValueError(f"Invalid framing_confidence: {obj.get('framing_confidence')!r}") from e
    if not 0<=conf<=1: raise ValueError(f"framing_confidence outside [0,1]: {conf}")
    raw_ev=obj.get("frame_evidence",[])
    evidence=[]; seen=set(); errs=[]
    if isinstance(raw_ev,list):
        for item in raw_ev:
            if not isinstance(item,dict): errs.append("non-object evidence item"); continue
            et=normalize_label(item.get("frame_type",item.get("frame_label"))); q=clean_text(item.get("quote"))
            if et not in types or not q: errs.append(f"invalid evidence item for {et!r}"); continue
            fixed=repair_evidence_quote(q,source_text)
            if not fixed: errs.append(f"ungrounded evidence for {et}: {q!r}"); continue
            for fq in fixed:
                if len(fq)<=MAX_EVIDENCE_CHARS:
                    evidence.append({"frame_type":et,"quote":fq}); seen.add(et)
    else: errs.append("frame_evidence was not a list")
    if types:
        for t in types:
            if t not in seen: errs.append(f"missing grounded evidence for {t}")
    else:
        evidence=[]; errs=[]
    is_not=not types
    return {"frame_label":label,"frame_type":types,"frame_explanation":explanation,"frame_evidence":evidence,
            "frame_evidence_grounded":(not types) or all(t in seen for t in types),"frame_evidence_error":"; ".join(errs),
            "framing_actors":normalize_actor_list(obj.get("framing_actors",["unspecified"]),"framing_actors",source_text,require_unspecified=is_not),
            "framed_actors":normalize_actor_list(obj.get("framed_actors",["unspecified"]),"framed_actors",source_text,require_unspecified=is_not),
            "framed_events":normalize_event_list(obj.get("framed_events",["unspecified"]),source_text,require_unspecified=is_not),
            "framing_confidence":conf,"frame_processing_error":False,"frame_processing_error_type":""}

def categorize_processing_error(exc: Exception) -> str:
    msg = str(exc)
    if isinstance(exc, requests.RequestException):
        return "OLLAMA_REQUEST_ERROR"
    if "done_reason=length" in msg or "generation truncated" in msg.lower():
        return "OLLAMA_TRUNCATION_ERROR"
    if any(x in msg for x in ("Empty model response", "No JSON object found", "Parsed JSON is not an object")):
        return "JSON_PARSE_ERROR"
    return "OUTPUT_VALIDATION_ERROR"


def call_evidence_repair(frame_types: List[str], title: str, segment_text: str) -> List[Dict[str, Any]]:
    prompt = f"The previous classification identified these frame types:\n{json.dumps(frame_types, ensure_ascii=False)}\n\nReturn ONLY a JSON object with key frame_evidence. For EACH frame type, provide one or more SHORT quotations copied as a single contiguous VERBATIM span from TITLE or SEGMENT_TEXT. Do not use ellipses, paraphrase, translation, or normalized characters.\n\nTITLE:\n{title}\n\nSEGMENT_TEXT:\n{segment_text}"
    payload={"model":OLLAMA_MODEL,"messages":[{"role":"user","content":prompt}],"stream":False,"format":"json",
             "options":{"temperature":0.0,"num_ctx":OLLAMA_NUM_CTX,"num_predict":1000}}
    payload["think"]=False
    r=requests.post(f"{OLLAMA_HOST}/api/chat",json=payload,timeout=1800); r.raise_for_status()
    data=r.json()
    if data.get("done_reason")=="length": raise ValueError("evidence repair truncated")
    obj=extract_json(data.get("message",{}).get("content",""))
    ev=obj.get("frame_evidence",[])
    return ev if isinstance(ev,list) else []

def call_actor_event_repair(frame_types: List[str], title: str, segment_text: str) -> Dict[str, List[str]]:
    prompt = f'''The previous framing classification identified these frame types:
{json.dumps(frame_types, ensure_ascii=False)}

Extract ONLY the following fields from the supplied source:
- framing_actors: who explicitly voices/advances/reports the framing
- framed_actors: who/what is the subject or target of the framing
- framed_events: the event/process/situation being framed

STRICT EXTRACTION RULES:
1. Every value other than "unspecified" MUST be copied as ONE SHORT CONTIGUOUS VERBATIM SPAN from TITLE or SEGMENT_TEXT.
2. Preserve original Chinese/historical/OCR characters exactly.
3. Do NOT translate into English, romanize, paraphrase, summarize, modernize characters, or add explanatory labels.
4. If no suitable span exists for a field, return ["unspecified"].
5. Return ONLY JSON with exactly these three keys.

TITLE:
{title}

SEGMENT_TEXT:
{segment_text}'''
    payload={"model":OLLAMA_MODEL,"messages":[{"role":"user","content":prompt}],"stream":False,"format":"json",
             "options":{"temperature":0.0,"num_ctx":OLLAMA_NUM_CTX,"num_predict":900}}
    payload["think"]=False
    r=requests.post(f"{OLLAMA_HOST}/api/chat",json=payload,timeout=1800); r.raise_for_status()
    data=r.json()
    if data.get("done_reason")=="length": raise ValueError("actor/event repair truncated")
    obj=extract_json(data.get("message",{}).get("content",""))
    out={}
    for field in ("framing_actors","framed_actors","framed_events"):
        val=obj.get(field,["unspecified"])
        if isinstance(val,str): val=[val]
        out[field]=val if isinstance(val,list) and val else ["unspecified"]
    return out


def call_model(title: str, segment_text: str) -> Dict[str, Any]:
    if len(segment_text) > MAX_CHARS:
        segment_text = segment_text[:MAX_CHARS]
    source_text = f"{title}\n{segment_text}"
    user = f"TITLE:\n{title}\n\nSEGMENT_TEXT:\n{segment_text}"
    payload: Dict[str, Any] = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        "stream": False,
        "format": "json",
        "options": {
            "temperature": TEMPERATURE,
            "num_ctx": OLLAMA_NUM_CTX,
            "num_predict": MAX_TOKENS,
        },
    }
    payload["think"] = OLLAMA_THINK

    last_err: Exception | None = None
    last_kind = "UNKNOWN_PROCESSING_ERROR"
    for attempt in range(MAX_RETRIES + 1):
        try:
            r = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=1800)
            if r.status_code >= 400 and "think" in payload:
                payload.pop("think", None)
                r = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=1800)
            r.raise_for_status()
            data = r.json()
            if data.get("done_reason") == "length":
                raise ValueError("Ollama generation truncated: done_reason=length")
            content = data.get("message", {}).get("content", "")
            raw_obj = extract_json(content)
            result = validate_and_repair(raw_obj, source_text)
            if result.get("frame_type") and not result.get("frame_evidence_grounded", False):
                try:
                    repaired_evidence = call_evidence_repair(result["frame_type"], title, segment_text)
                    repaired_obj = dict(raw_obj); repaired_obj["frame_evidence"] = repaired_evidence
                    repaired_result = validate_and_repair(repaired_obj, source_text)
                    if repaired_result.get("frame_evidence_grounded", False):
                        result = repaired_result
                        raw_obj = repaired_obj
                    else: result["frame_evidence_error"] += "; evidence-only repair did not fully ground all frame types"
                except Exception as repair_exc:
                    result["frame_evidence_error"] += f"; evidence-only repair failed: {type(repair_exc).__name__}: {repair_exc}"
            if result.get("frame_type") and extraction_needs_repair(raw_obj, source_text):
                try:
                    repaired_fields = call_actor_event_repair(result["frame_type"], title, segment_text)
                    patched_obj = dict(raw_obj)
                    patched_obj.update(repaired_fields)
                    if result.get("frame_evidence_grounded"):
                        patched_obj["frame_evidence"] = result.get("frame_evidence", [])
                    result = validate_and_repair(patched_obj, source_text)
                except Exception:
                    pass
            return result
        except Exception as e:
            last_err = e
            last_kind = categorize_processing_error(e)
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"{last_kind} after {MAX_RETRIES + 1} attempts: {last_err}")


def load_done_ids(jsonl_path: Path) -> set[str]:
    done: set[str] = set()
    if not jsonl_path.exists():
        return done
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                # Processing-error rows remain retryable on restart.
                if obj.get("_row_id") and not bool(obj.get("frame_processing_error", False)):
                    done.add(str(obj["_row_id"]))
            except Exception:
                continue
    return done


def read_jsonl(jsonl_path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    if not jsonl_path.exists():
        return rows
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except Exception:
                    pass
    return rows


def flatten_outputs(jsonl_path: Path, csv_path: Path, json_path: Path) -> None:
    rows = read_jsonl(jsonl_path)
    # Prefer the most recent successful record. If a row has never succeeded,
    # keep its most recent processing-error record for diagnosis.
    by_id: Dict[str, Dict[str, Any]] = {}
    for obj in rows:
        rid = str(obj.get("_row_id", ""))
        if not rid:
            continue
        prev = by_id.get(rid)
        obj_error = bool(obj.get("frame_processing_error", False))
        prev_error = bool(prev.get("frame_processing_error", False)) if prev else True
        if prev is None or prev_error or not obj_error:
            by_id[rid] = obj
    latest = list(by_id.values())
    latest.sort(key=lambda x: int(x.get("_input_row_number", 10**12)))

    json_path.write_text(json.dumps(latest, ensure_ascii=False, indent=2), encoding="utf-8")
    if not latest:
        pd.DataFrame().to_csv(csv_path, index=False)
        return

    csv_rows = []
    for obj in latest:
        flat = dict(obj)
        types = flat.get("frame_type", [])
        if isinstance(types, list):
            flat["frame_type"] = "|".join(types)
        evidence = flat.get("frame_evidence", [])
        if isinstance(evidence, list):
            flat["frame_evidence"] = json.dumps(evidence, ensure_ascii=False)
        for list_field in ("framing_actors", "framed_actors", "framed_events"):
            values = flat.get(list_field, [])
            if isinstance(values, list):
                flat[list_field] = "|".join(values)
        # Use literal TRUE/FALSE strings for easy filtering in spreadsheet software.
        flat["frame_processing_error"] = "TRUE" if bool(flat.get("frame_processing_error", False)) else "FALSE"
        flat["frame_evidence_grounded"] = "TRUE" if bool(flat.get("frame_evidence_grounded", False)) else "FALSE"
        csv_rows.append(flat)
    pd.DataFrame(csv_rows).to_csv(csv_path, index=False, encoding="utf-8-sig")


def main() -> None:
    ap = argparse.ArgumentParser(description="Detect predefined media frames in segmented newspaper articles using Ollama")
    ap.add_argument("--input", required=True, help="segmented/relevant CSV, e.g. sample_relevant.csv")
    ap.add_argument("--output-dir", required=True)
    ap.add_argument("--batch", type=int, default=None, help="maximum number of NEW segment rows to process this run")
    ap.add_argument("--output-stem", default=None, help="optional explicit stem for output filenames")
    args = ap.parse_args()

    input_path = Path(args.input)
    if not input_path.suffix:
        input_path = input_path.with_suffix(".csv")
    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model_slug = re.sub(r"[^A-Za-z0-9._-]+", "_", OLLAMA_MODEL)
    stem = args.output_stem or f"frames_{input_path.stem.removeprefix('segments_')}_{model_slug}"
    jsonl_path = out_dir / f"{stem}.jsonl"
    csv_path = out_dir / f"{stem}.csv"
    json_path = out_dir / f"{stem}.json"

    df = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    required = {"title", "segment_text"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Input CSV is missing required columns: {missing}. Found: {list(df.columns)}")

    work = df[df["segment_text"].astype(str).str.strip() != ""].copy()
    if "segmentation_label" in work.columns:
        work = work[work["segmentation_label"] != "NO_RELEVANT_SEGMENT"].copy()
    work["_input_row_number"] = work.index.astype(int)
    work["_row_id"] = work.apply(row_id, axis=1)

    done = load_done_ids(jsonl_path)
    todo = work[~work["_row_id"].isin(done)]
    if args.batch is not None:
        todo = todo.head(args.batch)

    print(f"[{now()}] model={OLLAMA_MODEL} host={OLLAMA_HOST}", flush=True)
    print(f"[{now()}] input relevant segment rows={len(work)} already successful={len(done)} this run={len(todo)}", flush=True)
    print(f"[{now()}] cumulative JSONL -> {jsonl_path}", flush=True)

    processed = 0
    with jsonl_path.open("a", encoding="utf-8") as jf:
        for _, row in todo.iterrows():
            rid = row["_row_id"]
            base = row.to_dict()
            try:
                result = call_model(clean_text(row.get("title")), clean_text(row.get("segment_text")))
                out = {
                    **base,
                    **result,
                    "processing_error": "",
                    "framing_model": OLLAMA_MODEL,
                }
                status = f"{out['frame_label']} {out.get('frame_type', [])} conf={out['framing_confidence']:.2f}"
            except Exception as e:
                out = {
                    **base,
                    "frame_label": "",
                    "frame_type": [],
                    "frame_explanation": "",
                    "frame_evidence": [],
                    "frame_evidence_grounded": False,
                    "frame_evidence_error": "",
                    "framing_actors": ["unspecified"],
                    "framed_actors": ["unspecified"],
                    "framed_events": ["unspecified"],
                    "framing_confidence": 0.0,
                    "frame_processing_error": True,
                    "frame_processing_error_type": str(e).split(" after ", 1)[0] if " after " in str(e) else "UNKNOWN_PROCESSING_ERROR",
                    "processing_error": str(e),
                    "framing_model": OLLAMA_MODEL,
                }
                status = f"PROCESSING_ERROR {e}"
            jf.write(json.dumps(out, ensure_ascii=False) + "\n")
            jf.flush()
            processed += 1
            print(f"[{now()}] {processed}/{len(todo)} {rid}: {status}", flush=True)
            if processed % CHECKPOINT_EVERY == 0:
                flatten_outputs(jsonl_path, csv_path, json_path)
                print(f"[{now()}] checkpoint -> {csv_path.name}, {json_path.name}", flush=True)

    flatten_outputs(jsonl_path, csv_path, json_path)
    print(f"[{now()}] done. outputs: {jsonl_path.name}, {csv_path.name}, {json_path.name}", flush=True)


if __name__ == "__main__":
    main()
"""

WORKER_SCRIPT_PATH.write_text(FRAME_DETECTION_PY, encoding="utf-8")
print(f"Wrote worker script: {WORKER_SCRIPT_PATH.resolve()}")


In [ ]:
# Inspect the selected sample before sending anything to Ollama.
import pandas as pd

sample_df = pd.read_csv(LOCAL_INPUT_CSV)
required = {"title", "segment_text"}
missing = required - set(sample_df.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"

print(f"Dataset: {LOCAL_INPUT_CSV}")
print(f"Rows: {len(sample_df):,}")
print(f"Non-empty segment_text rows: {sample_df['segment_text'].fillna('').astype(str).str.strip().ne('').sum():,}")
display(sample_df.head(5))


In [ ]:
# Sanity-check Ollama and run/resume the batch.
assert LOCAL_INPUT_CSV.exists(), f"Input CSV not found: {LOCAL_INPUT_CSV}"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    import requests
    r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
    r.raise_for_status()
    names = [m.get("name") for m in r.json().get("models", [])]
    print(f"Ollama reachable. Models available: {names}")
    if OLLAMA_MODEL not in names:
        print(f"WARNING: {OLLAMA_MODEL!r} is not in `ollama list`; check/pull the exact tag.")
except Exception as e:
    print(f"WARNING: Ollama check failed: {e}")

env = os.environ.copy()
env.update({
    "OLLAMA_HOST": OLLAMA_HOST,
    "OLLAMA_MODEL": OLLAMA_MODEL,
    "OLLAMA_NUM_CTX": str(OLLAMA_NUM_CTX),
    "OLLAMA_THINK": str(OLLAMA_THINK).lower(),
    "MAX_TOKENS": str(MAX_TOKENS),
    "TEMPERATURE": str(TEMPERATURE),
    "MAX_CHARS": str(MAX_CHARS),
    "CHECKPOINT_EVERY": str(CHECKPOINT_EVERY),
    "MAX_RETRIES": str(MAX_RETRIES),
    "MAX_EVIDENCE_CHARS": str(MAX_EVIDENCE_CHARS),
})

paths = output_paths()
cmd = [sys.executable, str(WORKER_SCRIPT_PATH), "--input", str(LOCAL_INPUT_CSV), "--output-dir", str(LOCAL_OUTPUT_DIR)]
if BATCH_SIZE is not None:
    cmd += ["--batch", str(BATCH_SIZE)]

print("Running:", " ".join(cmd))
print("Cumulative JSONL:", paths["jsonl"])

if RUN_IN_BACKGROUND:
    log_f = open(paths["log"], "a", buffering=1)
    proc = subprocess.Popen(cmd, env=env, stdout=log_f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
    print(f"Started detached worker PID={proc.pid}; log={paths['log'].resolve()}")
else:
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print("exit code:", proc.returncode)

In [ ]:
# Optional: watch a detached run. Interrupting this cell does NOT stop the worker.
import time
log_path = output_paths()["log"]
if not log_path.exists():
    print("No log yet:", log_path)
else:
    with open(log_path, "r", encoding="utf-8") as f:
        f.seek(0, 2)
        try:
            while True:
                line = f.readline()
                if line:
                    print(line, end="")
                else:
                    time.sleep(2)
        except KeyboardInterrupt:
            print("\nStopped watching; worker is unaffected.")

In [ ]:
# Progress and quick distribution check.
import json
import pandas as pd

paths = output_paths()
jsonl_path, csv_path = paths["jsonl"], paths["csv"]

if jsonl_path.exists():
    records = []
    for line in jsonl_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            try:
                records.append(json.loads(line))
            except Exception:
                pass
    successful = {
        r.get("_row_id") for r in records
        if r.get("_row_id") and not bool(r.get("frame_processing_error", False))
    }
    errors = [r for r in records if bool(r.get("frame_processing_error", False))]
    print(f"Successful unique segment rows: {len(successful)}; logged processing-error records: {len(errors)}")
else:
    print("No JSONL yet.")

if csv_path.exists():
    df = pd.read_csv(csv_path, keep_default_na=False)
    if "frame_processing_error" in df.columns:
        err = df["frame_processing_error"].astype(str).str.upper().eq("TRUE")
        ok = df[~err].copy()
        print(f"Current flattened rows: {len(df)}; processing errors: {err.sum()}")
    else:
        ok = df.copy()

    if len(ok):
        exploded = (
            ok.assign(frame_label=ok["frame_label"].fillna("").str.split("|", regex=False))
              .explode("frame_label")
        )
        exploded = exploded[exploded["frame_label"] != ""]
        print("\nframe_label counts (multi-frame rows contribute to each label):")
        print(exploded["frame_label"].value_counts(dropna=False))
        print("\nNOT_FRAMED share:", (ok["frame_label"] == "NOT_FRAMED").mean())
        print("\nSample (including actor/event attribution):")
        actor_cols = [c for c in ["title", "segment_text", "frame_label", "framing_actors", "framed_actors", "framed_events", "frame_evidence", "framing_confidence"] if c in ok.columns]
        display(ok.sample(min(5, len(ok)))[actor_cols])
    else:
        print("No successful rows yet.")
else:
    print("No flattened CSV yet.")

# Diagnostic breakdown for processing failures (v5).
if csv_path.exists():
    _diag = pd.read_csv(csv_path)
    if "frame_processing_error_type" in _diag.columns:
        bad = _diag[_diag["frame_processing_error"].astype(str).str.upper().eq("TRUE")]
        if len(bad):
            print("\nProcessing-error types:")
            print(bad["frame_processing_error_type"].fillna("UNKNOWN").value_counts())


In [ ]:
# MODEL COMPARISON on the exact same fixed segment-row sample.
# Each model writes to an isolated directory, so this does not affect full-corpus outputs.
import shutil
import subprocess
import pandas as pd

MODELS_TO_COMPARE = [
    "qwen3.5:35b",
    "llama3.3:70b",
    # "qwen3:30b",
    # "gemma3:27b",
]
COMPARISON_SAMPLE_SIZE = 50
COMPARISON_DIR = LOCAL_OUTPUT_DIR / "frame_model_comparison"
CLEAN_COMPARISON_DIR = True

if CLEAN_COMPARISON_DIR and COMPARISON_DIR.exists():
    shutil.rmtree(COMPARISON_DIR)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
WORKER_SCRIPT_PATH.write_text(FRAME_DETECTION_PY, encoding="utf-8")

results = {}
for model in MODELS_TO_COMPARE:
    out = COMPARISON_DIR / model_slug(model)
    out.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env.update({
        "OLLAMA_HOST": OLLAMA_HOST, "OLLAMA_MODEL": model,
        "OLLAMA_NUM_CTX": str(OLLAMA_NUM_CTX), "OLLAMA_THINK": "false",
        "MAX_TOKENS": str(MAX_TOKENS), "TEMPERATURE": str(TEMPERATURE),
        "MAX_CHARS": str(MAX_CHARS), "CHECKPOINT_EVERY": str(CHECKPOINT_EVERY),
        "MAX_RETRIES": str(MAX_RETRIES), "MAX_EVIDENCE_CHARS": str(MAX_EVIDENCE_CHARS),
    })
    stem = f"frames_{DATASET_STEM}_{model_slug(model)}"
    cmd = [sys.executable, str(WORKER_SCRIPT_PATH), "--input", str(LOCAL_INPUT_CSV), "--output-dir", str(out),
           "--batch", str(COMPARISON_SAMPLE_SIZE), "--output-stem", stem]
    print(f"\n=== {model} ===")
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    f = out / f"{stem}.csv"
    results[model] = pd.read_csv(f, keep_default_na=False) if f.exists() else None

available = {m:d for m,d in results.items() if d is not None}
if len(available) >= 2:
    indexed = {}
    for m, d in available.items():
        if "frame_processing_error" in d.columns:
            x = d[~d["frame_processing_error"].astype(str).str.upper().eq("TRUE")].copy()
        else:
            x = d.copy()
        keep_cols = [c for c in ["frame_label", "frame_type", "framing_actors", "framed_actors", "framed_events", "framing_confidence"] if c in x.columns]
        indexed[m] = x.set_index("_row_id")[keep_cols]

    common = None
    for x in indexed.values():
        common = x.index if common is None else common.intersection(x.index)

    label_table = pd.DataFrame({m:x.loc[common, "frame_label"] for m,x in indexed.items()})
    label_agree = label_table.nunique(axis=1) == 1
    print(f"\nCommon successful rows: {len(common)}")
    print(f"Exact frame_label agreement: {label_agree.mean():.1%}")
    type_table = pd.DataFrame({m:x.loc[common, "frame_type"] for m,x in indexed.items()})
    type_agree = type_table.nunique(axis=1) == 1
    print(f"Exact frame_type agreement: {type_agree.mean():.1%}")
    print("\nDisagreements:")
    display(label_table[~label_agree].head(50))
else:
    print("Need at least two successful model conditions.")
